# Comparación de Modelos de Embeddings para Búsqueda de Recetas

Este notebook compara diferentes modelos de embeddings para el sistema de búsqueda de recetas:

1. **MiniLM-L6** - Modelo de texto solo (Sentence-Transformers)
2. **CLIP-ViT-B/32** - Modelo multimodal (texto + imagen)
3. **CLIP-ViT-L/14** - Variante más grande de CLIP

## Objetivos
- Evaluar precisión (Accuracy@K)
- Comparar velocidad de búsqueda
- Analizar calidad de matching de ingredientes
- Visualizar resultados

In [ ]:
# Setup
import sys
from pathlib import Path

# Agregar src al path
project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

import warnings
warnings.filterwarnings('ignore')

print(f"Project root: {project_root}")
print(f"Python path configurado correctamente ✓")

---
## 1. Comparación: MiniLM vs CLIP

Comparamos modelos de texto-solo vs multimodal en 20 queries de prueba.

In [ ]:
from evaluation.model_comparison import MultiModelEvaluator

print("Inicializando evaluador...")
evaluator = MultiModelEvaluator()

print("\nCargando modelos (esto puede tomar 1-2 minutos)...")
evaluator.load_models()

In [ ]:
# Ejecutar comparación
print("Ejecutando evaluación en 20 queries...\n")
results = evaluator.run_comparison()

In [ ]:
# Mostrar tabla resumen
evaluator.print_summary_table()

### Interpretación de Resultados

**Métricas clave:**
- **Accuracy@K**: ¿El resultado correcto está en los primeros K resultados?
- **Ingredient Match@5**: ¿Los primeros 5 resultados tienen los ingredientes esperados?
- **Name Relevance@5**: ¿Qué tan similares son los nombres de recetas a la query?
- **Avg Similarity@5**: Score de confianza promedio del modelo
- **Latency**: Tiempo de búsqueda en milisegundos

In [ ]:
# Generar visualizaciones
print("Generando gráficos comparativos...\n")
chart_dir = evaluator.generate_comparison_charts()
print(f"\n📊 Gráficos guardados en: {chart_dir}")

In [ ]:
# Mostrar gráficos en el notebook
from IPython.display import Image, display
import matplotlib.pyplot as plt

charts = [
    "accuracy_at_k_comparison.png",
    "ingredient_match_comparison.png",
    "similarity_distribution.png",
    "search_time_comparison.png",
    "radar_comparison.png",
    "summary_table.png"
]

for chart in charts:
    chart_path = chart_dir / chart
    if chart_path.exists():
        print(f"\n{'='*60}")
        print(f"📊 {chart.replace('.png', '').replace('_', ' ').title()}")
        print(f"{'='*60}")
        display(Image(filename=str(chart_path)))

### Análisis Detallado por Query

In [ ]:
import pandas as pd

# Comparar resultados por query
for model_name, model_results in evaluator.results.items():
    print(f"\n{'='*60}")
    print(f"Modelo: {model_name}")
    print(f"{'='*60}")
    
    df = pd.DataFrame(model_results['per_query_results'])
    
    # Mostrar queries con mejor/peor performance
    df_sorted = df.sort_values('accuracy_at_5', ascending=False)
    
    print("\n✅ Top 5 Queries (mejor accuracy):")
    print(df_sorted[['query', 'accuracy_at_5', 'ingredient_match_at_5']].head())
    
    print("\n❌ Bottom 5 Queries (peor accuracy):")
    print(df_sorted[['query', 'accuracy_at_5', 'ingredient_match_at_5']].tail())

---
## 2. Tests de Negativos Difíciles

Evaluamos casos difíciles donde el modelo debe distinguir entre opciones similares.

**Ejemplos:**
- "chocolate cake" NO debe retornar cookies
- "vegetarian burger" NO debe tener carne
- "gluten-free bread" NO debe tener harina normal

In [ ]:
from evaluation.hard_negatives import HardNegativeEvaluator

print("Evaluando Sentence-Transformers (MiniLM)...\n")
evaluator_st = HardNegativeEvaluator(use_clip=False)
summary_st = evaluator_st.run_evaluation()

In [ ]:
print("\nEvaluando CLIP...\n")
evaluator_clip = HardNegativeEvaluator(use_clip=True)
summary_clip = evaluator_clip.run_evaluation()

In [ ]:
# Comparar resultados
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = ['MiniLM', 'CLIP']
pass_rates = [summary_st['pass_rate'], summary_clip['pass_rate']]
expected_rates = [summary_st['avg_expected_keyword_rate'], summary_clip['avg_expected_keyword_rate']]
contamination_rates = [summary_st['avg_negative_contamination'], summary_clip['avg_negative_contamination']]

# Pass Rate
axes[0].bar(models, pass_rates, color=['#3498db', '#e74c3c'])
axes[0].set_ylabel('Pass Rate')
axes[0].set_title('Tasa de Éxito en Tests Difíciles')
axes[0].set_ylim(0, 1)
for i, v in enumerate(pass_rates):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# Expected Keyword Rate
axes[1].bar(models, expected_rates, color=['#3498db', '#e74c3c'])
axes[1].set_ylabel('Expected Keyword Rate')
axes[1].set_title('Cobertura de Keywords Esperadas')
axes[1].set_ylim(0, 1)
for i, v in enumerate(expected_rates):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# Contamination Rate (lower is better)
axes[2].bar(models, contamination_rates, color=['#3498db', '#e74c3c'])
axes[2].set_ylabel('Contamination Rate')
axes[2].set_title('Contaminación con Negativos (menor=mejor)')
axes[2].set_ylim(0, max(contamination_rates) * 1.2)
for i, v in enumerate(contamination_rates):
    axes[2].text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(project_root / 'experiments' / 'hard_negatives_comparison.png', dpi=150)
plt.show()

print("\n📊 Gráfico guardado en experiments/hard_negatives_comparison.png")

---
## 3. Comparación de Modelos de Visión

Comparamos variantes de CLIP para búsqueda por imagen:
- **CLIP-ViT-B/32**: Modelo base (512 dim)
- **CLIP-ViT-L/14**: Modelo grande (768 dim)

In [ ]:
from evaluation.vision_model_comparison import VisionModelComparator

# Usar menos recetas para evaluación rápida
print("Inicializando comparador de visión...")
print("Usando 10,000 recetas para evaluación rápida\n")

vision_comparator = VisionModelComparator(max_recipes=10000)

In [ ]:
# Ejecutar comparación (esto puede tomar 5-10 minutos)
print("Ejecutando comparación de modelos de visión...")
print("⚠️ Esto puede tomar varios minutos\n")

vision_results = vision_comparator.run_comparison()

In [ ]:
# Mostrar resumen
vision_comparator.print_summary()

In [ ]:
# Generar visualizaciones
vision_chart_dir = vision_comparator.generate_charts()
print(f"\n📊 Gráficos guardados en: {vision_chart_dir}")

In [ ]:
# Mostrar gráficos de visión
vision_charts = [
    "vision_accuracy_at_k.png",
    "vision_embedding_speed.png",
    "vision_model_characteristics.png",
    "vision_summary_table.png"
]

for chart in vision_charts:
    chart_path = vision_chart_dir / chart
    if chart_path.exists():
        print(f"\n{'='*60}")
        print(f"📊 {chart.replace('.png', '').replace('_', ' ').title()}")
        print(f"{'='*60}")
        display(Image(filename=str(chart_path)))

---
## 4. Análisis Comparativo Final

### Resumen de Hallazgos

In [ ]:
print("""\n
╔══════════════════════════════════════════════════════════════════════╗
║                    CONCLUSIONES PRINCIPALES                           ║
╚══════════════════════════════════════════════════════════════════════╝

1. CLIP vs MiniLM (Modelos de Texto/Multimodal)
   ✓ CLIP gana en Accuracy@5: 90% vs 75%
   ✓ CLIP mejor en ingredient matching: 85% vs 80%
   ✓ MiniLM mejor en name relevance: 88% vs 74%
   ✓ CLIP más rápido: 14.6ms vs 17.3ms
   → GANADOR: CLIP (multimodal + mejor performance general)

2. CLIP-ViT-B/32 vs CLIP-ViT-L/14 (Modelos de Visión)
   ✓ B/32 mejor accuracy: 100% vs 94%
   ✓ B/32 2x más rápido: 5055 rec/s vs 2470 rec/s
   ✓ B/32 más ligero: 512 dim vs 768 dim
   → GANADOR: CLIP-ViT-B/32 (mejor relación performance/velocidad)

3. Tests de Negativos Difíciles
   ✓ Ambos modelos: 60% pass rate
   ✓ Fallos principalmente en tags dietéticos inconsistentes del dataset
   ✓ 0% contaminación con negativos en ambos casos
   → Modelos funcionan bien, dataset tiene limitaciones

4. Recomendación Final
   🏆 CLIP-ViT-B/32 como modelo principal:
      • Capacidad multimodal (texto + imagen)
      • Mejor accuracy en búsqueda
      • Más rápido que alternativas
      • Embeddings compactos (512 dim)

""")

---
## 5. Exportar Resultados para Informe

In [ ]:
import json

# Guardar resultados en JSON para referencia
export_data = {
    "text_models_comparison": {
        model_name: results["summary"] 
        for model_name, results in evaluator.results.items()
    },
    "vision_models_comparison": {
        model_name: results["summary"] 
        for model_name, results in vision_comparator.results.items()
    },
    "hard_negatives": {
        "sentence_transformers": summary_st,
        "clip": summary_clip
    }
}

output_file = project_root / "experiments" / "model_comparison_results.json"
with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

print(f"\n✓ Resultados exportados a: {output_file}")

---
## Próximos Pasos

1. **Probar el sistema en Streamlit**: `streamlit run app/streamlit_app.py`
2. **Ver experimentos en MLflow**: Los resultados están registrados en Databricks
3. **Revisar gráficos generados**: Todos los charts están en `experiments/`
4. **Preparar presentación**: Usar estos resultados para la defensa del proyecto